# Pitch Profiler’s Iteration of Dynamic Dead Zone

This implementation is inspired by Max Bay’s Dynamic Dead Zone with several key differences:  
- It analyzes all pitch types, not just fastballs.  
- It employs eight distinct LightGBM models to generate the dynamic dead zones.  
- It applies a separate dead-zone selection algorithm (Hungarian Assignment) for each pitch type.  

You can explore Max’s Dynamic Dead Zone [here](https://dynamic-dead-zone.streamlit.app/) and follow him on X [here](https://x.com/choice_fielder).

#### *Copyright Pending*

Under U.S. copyright law, you may:  
- Use the ideas, methods, or procedures behind the work, provided you write your own implementation (e.g., a “clean-room” reimplementation of an algorithm).  
- Independently create a work that is similar in function, as long as you do not copy protected expression.  
- Quote limited excerpts for criticism, commentary, news reporting, scholarship, or research (fair use).  
- Create transformative or parody works that add new expression, meaning, or message (fair use).  
- Reverse-engineer software for interoperability or security research, subject to applicable statutory exceptions (e.g., DMCA provisions).  
- Make archival or backup copies of lawfully acquired works.  
- Perform private viewing, playing, or execution of legitimately acquired software.  
- Link to or embed publicly available code or media, provided you do not distribute the files themselves.  

Under U.S. copyright law, you may **not**:  
- Reproduce source code, object code, documentation, graphics, UI layouts, or other protected expression without authorization.  
- Distribute copies—sell, rent, host, upload, or otherwise make the work publicly available.  
- Prepare derivative works—adapt, translate, modify, or refactor the protected content without permission.  
- Perform or display the work publicly (e.g., demo the app at a paid event or stream the software to an audience) without authorization.  
- Circumvent technological protection measures (DRM) applied to the work.  
- Publish substantial excerpts that together constitute the “heart” of the work absent fair-use justification.  

###### You may use this code, Max’s implementation, or other public implementations of Dynamic Dead Zone (or expected movement) for inspiration, but you may **not** copy the code and sell it as your own.

In [1]:
# Import the packages
import polars as pl
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
import requests
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler
from lightgbm import LGBMRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('max_colwidth', None)


Load in Model Data - You can substitute this in with any Baseball Savant scraper

In [ ]:
#load_data_for_year will not work as the proprietary link to pitch profiler's training api is removed
def load_data_for_year(game_year: int) -> list:
    """
    Loads data from the JSON URL for a given game year and returns a list of dictionaries.
    
    Parameters:
        game_year (int): The game year to load data for.
        
    Returns:
        List of dictionaries containing the data, or an empty list on error.
    """
    url = '' #proprietary Pitch Profiler API
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raise for HTTP errors
        data = response.json()
        
        # If the data is nested in an "items" key, use that
        if isinstance(data, dict) and "items" in data:
            return data["items"]
        # Otherwise assume data is directly a list of records
        elif isinstance(data, list):
            return data
        else:
            print(f"Unexpected data format for year {game_year}")
            return []
    except requests.RequestException as e:
        print(f"Error fetching data for year {game_year}: {e}")
        return []
    except Exception as e:
        print(f"Error processing data for year {game_year}: {e}")
        return []

# List to hold all records from all years
all_records = []

# Iterate from 2024 down to 2020 (inclusive)
for year in range(2024, 2019, -1):
    print(f"Fetching data for game year {year}...")
    records = load_data_for_year(year)
    if records:
        all_records.extend(records)  # Append records to our master list

if all_records:
    df = pl.DataFrame(all_records, infer_schema_length=len(all_records))
else:
    print("No data loaded.")

df.describe()

Fetching data for game year 2024...
Fetching data for game year 2023...
Fetching data for game year 2022...
Fetching data for game year 2021...
Fetching data for game year 2020...


statistic,game_year,game_date,game_pk,pitcher_name,pitcher,pitch_type,p_throws,stand,hand_split,inning,at_bat_number,pitch_number,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,n_pitches_pitch_type_batter_game,n_pitches_pitch_type_batter_at_bat,balls,strikes,primary_fastball,release_speed,fastball_release_speed_diff,release_spin_rate,vb,fastball_vb_diff,amagx_normalized,fastball_amagx_normalized_diff,amagz,fastball_amagz_diff,armslot_amagx_normalized,armslot_amagz,tf,fastball_tf_diff,release_extension,release_pos_x_normalized,release_pos_z,arm_angle_normalized,plate_x_normalized,plate_z_normalized,vra,prev_vra_diff,hra_normalized,prev_hra_normalized_diff,rv,whiff
str,f64,str,f64,str,f64,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",2.969123e6,"""2969123""",2.969123e6,"""2969123""",2.969123e6,"""2969123""","""2969123""","""2969123""","""2969123""",2.969123e6,2.969123e6,2.969123e6,2.969123e6,2.969123e6,2.969123e6,2.969123e6,2.969123e6,2.969123e6,2.969123e6,2.969123e6,1.564218e6,2.969123e6,2.969111e6,1.564213e6,2.969123e6,1.564218e6,2.969123e6,1.564218e6,2.969123e6,2.969123e6,2.969123e6,1.564218e6,2.969123e6,2.969123e6,2.969123e6,2.969123e6,2.969123e6,2.969123e6,2.969123e6,2.211523e6,2.969123e6,2.211523e6,2.969123e6,2.969123e6
"""null_count""",0.0,"""0""",0.0,"""0""",0.0,"""0""","""0""","""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.404905e6,0.0,12.0,1.40491e6,0.0,1.404905e6,0.0,1.404905e6,0.0,0.0,0.0,1.404905e6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,757600.0,0.0,757600.0,0.0,0.0
"""mean""",2022.641983,null,697082.924363,null,620039.18799,null,null,null,null,4.832267,37.277236,2.926466,1.496503,1.463787,2.919951,2.010345,0.888852,0.902246,0.473128,89.266995,8.207658,2259.187111,-27.184991,17.719482,7.165478,10.367842,6.888987,11.941103,-0.902082,10.196171,0.423422,-0.040087,6.407364,-1.88113,5.794816,50.929585,0.063501,2.291467,-1.340823,0.027736,2.348486,-0.003695,-0.00069,0.124349
"""std""",1.300861,null,47044.526029,null,61426.65256,null,null,null,null,2.581887,22.243772,1.743023,0.713845,1.230875,2.514697,1.494996,0.971535,0.828309,0.499277,5.872247,4.679924,337.554203,12.803411,12.311662,10.136436,11.57407,8.855091,8.80676,5.574371,12.004704,0.0299,0.024989,0.446126,0.727839,0.541034,12.940634,0.825283,1.083035,1.399296,1.588508,1.185956,1.186872,0.144477,0.329979
"""min""",2020.0,"""01-APR-21""",630099.0,"""A.J. Alexy""",424144.0,"""CH""","""L""","""L""","""OHH""",1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,53.0,-9.3,16.0,-87.84,-27.16,-25.031097,-30.204327,-21.541523,-27.102686,-21.011034,-23.960308,0.351799,-0.189958,4.2,-4.84,0.81,0.0,-4.94,-6.7,-8.558244,-13.674404,-3.750655,-11.065258,-0.390427,0.0
"""25%""",2022.0,null,661119.0,null,593423.0,null,null,null,null,3.0,18.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,85.0,5.6,2107.0,-35.4,10.4,-1.469319,2.462966,0.425574,6.746166,-5.281217,-0.068731,0.400146,-0.055119,6.1,-2.35,5.53,42.7,-0.5,1.59,-2.287296,-0.957651,1.553018,-0.768911,-0.059595,0.0
"""50%""",2023.0,null,717194.0,null,641703.0,null,null,null,null,5.0,36.0,3.0,1.0,1.0,2.0,1.0,1.0,1.0,0.0,90.3,8.5,2283.0,-25.44,16.94,8.88517,11.365424,6.966735,12.085053,-1.725803,14.959158,0.416389,-0.039765,6.4,-1.85,5.84,49.7,0.06,2.3,-1.432796,0.005564,2.33157,0.003714,-0.000084,0.0
"""75%""",2024.0,null,745547.0,null,666142.0,null,null,null,null,7.0,56.0,4.0,2.0,2.0,4.0,3.0,2.0,2.0,1.0,94.0,11.3,2455.0,-16.08,24.28,15.494546,18.59634,14.904465,17.451589,3.446465,20.028541,0.442789,-0.024822,6.7,-1.39,6.13,57.7,0.62,3.01,-0.499853,0.988331,3.127893,0.770632,0.061051,0.0
"""max""",2024.0,"""31-MAY-24""",747224.0,"""Zebby Matthews""",814005.0,"""SV""","""R""","""R""","""SHH""",16.0,125.0,19.0,4.0,6.0,33.0,25.0,4.0,3.0,1.0,105.1,27.6,3722.0,-1.2,68.62,39.316382,50.34715,30.562235,43.512999,20.881417,40.195505,0.728328,0.049448,8.3,1.12,7.46,160.7,5.11,12.66,8.631743,14.14462,9.491601,9.544531,1.737614,1.0


In [3]:
stats_by_pitch_type = (
    df.group_by("pitch_type")
    .agg(
        pl.col("game_date").min().alias("min_game_date"),
        pl.count().alias("count")
    )
    .sort("count", descending=True)  # Sort by count in descending order
)
print(stats_by_pitch_type.to_pandas().to_string())

C:\Users\jerem\AppData\Local\Temp\ipykernel_23244\3890644441.py:5: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
  pl.count().alias("count")


   pitch_type min_game_date    count
0          FF     01-APR-21  1031746
1          SL     01-APR-21   486475
2          SI     01-APR-21   466903
3          CH     01-APR-21   291819
4          FC     01-APR-21   228049
5          CU     01-APR-21   194474
6          ST     01-APR-21   128706
7          FS     01-APR-21    67070
8          KC     01-APR-21    59932
9          SV     01-APR-23    11240
10         KN     01-APR-24     1668
11         FO     01-SEP-23      648
12         SC     02-SEP-24      262
13         FA     02-SEP-22      131


Seperating into models to train on

In [4]:
# Create separate DataFrames for each pitch type
cutter_df    = df.filter(pl.col("pitch_type").is_in(["FC"])) 
slider_df    = df.filter(pl.col("pitch_type").is_in(["SL"])) 
four_seam_df = df.filter(pl.col("pitch_type") == "FF")
sinker_df    = df.filter(pl.col("pitch_type") == "SI")
sweeper_df    = df.filter(pl.col("pitch_type").is_in(["ST", 'SV']))  # Distinct slider DF
curve_df     = df.filter(pl.col("pitch_type").is_in(["KC", "CU", "CS"]))  # Distinct curve DF
offspeed_df  = df.filter(pl.col("pitch_type").is_in(["CH", "FS", "SC", "FO", "KN"]))

four_seam_df.describe()

statistic,game_year,game_date,game_pk,pitcher_name,pitcher,pitch_type,p_throws,stand,hand_split,inning,at_bat_number,pitch_number,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,n_pitches_pitch_type_batter_game,n_pitches_pitch_type_batter_at_bat,balls,strikes,primary_fastball,release_speed,fastball_release_speed_diff,release_spin_rate,vb,fastball_vb_diff,amagx_normalized,fastball_amagx_normalized_diff,amagz,fastball_amagz_diff,armslot_amagx_normalized,armslot_amagz,tf,fastball_tf_diff,release_extension,release_pos_x_normalized,release_pos_z,arm_angle_normalized,plate_x_normalized,plate_z_normalized,vra,prev_vra_diff,hra_normalized,prev_hra_normalized_diff,rv,whiff
str,f64,str,f64,str,f64,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",1.031746e6,"""1031746""",1.031746e6,"""1031746""",1.031746e6,"""1031746""","""1031746""","""1031746""","""1031746""",1.031746e6,1.031746e6,1.031746e6,1.031746e6,1.031746e6,1.031746e6,1.031746e6,1.031746e6,1.031746e6,1.031746e6,1.031746e6,80781.0,1.031746e6,1.031737e6,80778.0,1.031746e6,80781.0,1.031746e6,80781.0,1.031746e6,1.031746e6,1.031746e6,80781.0,1.031746e6,1.031746e6,1.031746e6,1.031746e6,1.031746e6,1.031746e6,1.031746e6,759888.0,1.031746e6,759888.0,1.031746e6,1.031746e6
"""null_count""",0.0,"""0""",0.0,"""0""",0.0,"""0""","""0""","""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,950965.0,0.0,9.0,950968.0,0.0,950965.0,0.0,950965.0,0.0,0.0,0.0,950965.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,271858.0,0.0,271858.0,0.0,0.0
"""mean""",2022.579763,null,694994.513775,null,623960.992435,null,null,null,null,4.776065,36.825001,2.950446,1.448718,1.4293,3.382971,2.243127,0.940353,0.876777,0.921705,94.080819,-1.040092,2291.643673,-14.918636,-6.611286,11.043673,4.265883,16.052961,-6.414327,-4.878336,19.143233,0.39993,0.004526,6.46929,-1.830931,5.859168,48.683421,0.01975,2.856861,-1.974271,0.467119,2.468221,-0.131144,0.000741,0.109027
"""std""",1.322679,null,47486.84449,null,60272.302689,null,null,null,null,2.644412,22.86936,1.773865,0.687696,1.256229,2.92037,1.678743,1.020711,0.835324,0.268636,2.477089,1.777361,166.584135,3.341021,3.672305,5.064812,7.730481,3.196723,3.377233,3.116042,3.944662,0.011391,0.007909,0.439437,0.713149,0.479069,11.569262,0.747595,0.94985,1.137883,1.50678,1.127672,1.113906,0.150495,0.311673
"""min""",2020.0,"""01-APR-21""",630099.0,"""A.J. Alexy""",425794.0,"""FF""","""L""","""L""","""OHH""",1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,82.3,-9.3,1376.0,-44.88,-23.02,-10.878308,-23.35726,-8.391665,-20.325251,-20.525711,-3.051821,0.356797,-0.061384,4.3,-4.84,1.87,0.0,-3.89,-2.23,-7.888864,-13.674404,-2.380021,-9.245491,-0.390427,0.0
"""25%""",2021.0,null,634476.0,null,595465.0,null,null,null,null,2.0,17.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,92.5,-1.9,2182.0,-16.8,-8.88,7.701049,2.188347,14.258455,-8.625644,-7.039973,17.036834,0.392163,-0.000767,6.2,-2.3,5.6,40.9,-0.48,2.22,-2.732073,-0.549469,1.702127,-0.843821,-0.059595,0.0
"""50%""",2023.0,null,716954.0,null,642207.0,null,null,null,null,5.0,36.0,3.0,1.0,1.0,3.0,2.0,1.0,1.0,1.0,94.1,-0.7,2294.0,-14.52,-6.36,11.108372,6.567889,16.363322,-6.297058,-5.023439,19.520414,0.399626,0.003149,6.5,-1.81,5.88,47.9,0.03,2.84,-2.002173,0.351027,2.442576,-0.10547,-0.000084,0.0
"""75%""",2024.0,null,745491.0,null,668881.0,null,null,null,null,7.0,56.0,4.0,2.0,2.0,5.0,3.0,2.0,2.0,1.0,95.7,0.2,2404.0,-12.6,-4.16,14.467854,9.373658,18.230136,-4.184852,-2.860349,21.746704,0.407146,0.0085,6.8,-1.34,6.16,55.3,0.53,3.48,-1.253907,1.349658,3.206323,0.612126,0.061051,0.0
"""max""",2024.0,"""31-MAY-24""",747224.0,"""Zebby Matthews""",814005.0,"""FF""","""R""","""R""","""SHH""",15.0,121.0,19.0,4.0,6.0,33.0,25.0,4.0,2.0,1.0,104.8,8.0,3212.0,-3.0,10.34,32.831056,26.671201,28.521235,10.528196,13.612305,36.380586,0.466384,0.049448,8.3,0.73,7.4,124.9,4.18,12.66,6.337897,13.277612,8.239425,9.544531,1.737614,1.0


Define features and target

In [5]:
# Define the features and target variable
features = [
    'release_pos_z', 'arm_angle_normalized', 'release_speed'
]
target = ['amagx_normalized', 'amagz']

# Convert Polars DataFrames to Pandas
cutter_df = cutter_df.to_pandas()
slider_df = slider_df.to_pandas()
four_seam_df = four_seam_df.to_pandas()
sinker_df = sinker_df.to_pandas()
sweeper_df = sweeper_df.to_pandas()
curve_df = curve_df.to_pandas()
offspeed_df = offspeed_df.to_pandas()


Create the Model Function

In [7]:
def train_and_evaluate_with_cv(df, features, target, param_grid, n_iter=20, cv=5):
    """
    Train and evaluate a multi-output model using cross-validation and hyperparameter tuning.

    Parameters:
    - df: DataFrame (Pandas or Polars) containing the data.
    - features: list of feature column names.
    - target: list of target column names.
    - param_grid: dictionary of hyperparameter ranges for LightGBM (applied to the estimator).
    - n_iter: number of parameter combinations to test.
    - cv: number of cross-validation folds.

    Returns:
    - best_model: trained multi-output LightGBM model with the best parameters.
    - best_params: dictionary of best hyperparameters.
    - best_score: best CV score (converted to a positive RMSE value).
    """
    # Extract features (X) and multi-output targets (y)
    X = df[features]
    y = df[target]
    
    # Identify numeric features for scaling (example: exclude any specific non-numeric or categorical features if needed)
    numeric_features = [col for col in features if col != 'hand_split']

    # Create a ColumnTransformer for scaling numeric features
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', RobustScaler(), numeric_features),
        ],
        remainder='passthrough'  # Keep remaining columns (e.g., 'hand_split') unchanged
    )
    
    # Wrap LightGBM in a MultiOutputRegressor
    multioutput_regressor = MultiOutputRegressor(
        LGBMRegressor(random_state=42, force_row_wise=True)
    )
    
    # Create the pipeline: first the preprocessor then the multi-output model.
    pipeline = make_pipeline(
        preprocessor,
        multioutput_regressor
    )
    
    # Update parameter grid keys to target the underlying estimator
    # e.g., param 'n_estimators' becomes 'multioutputregressor__estimator__n_estimators'
    tuned_param_grid = {
        'multioutputregressor__estimator__' + k: v for k, v in param_grid.items()
    }
    
    # Setup RandomizedSearchCV with negative mean squared error as scoring
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=tuned_param_grid,
        n_iter=n_iter,
        cv=cv,
        verbose=2,
        scoring='neg_mean_squared_error',
        random_state=42,
        n_jobs=-1
    )
    
    # Fit the search object
    search.fit(X, y)
    
    # Extract best estimator, best parameters and the best score (convert negative MSE to positive RMSE)
    best_model = search.best_estimator_
    best_params = search.best_params_
    best_score = (-search.best_score_) ** 0.5  # converting MSE to RMSE
    
    return best_model, best_params, best_score

Model - this takes about 12 hours

In [8]:
param_grid = {
    'n_estimators': [1500, 2000, 2500],  # Number of boosting rounds
    'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],  # Learning rate (smaller values for more rounds)
    'num_leaves': [31, 50, 70, 100, 120],  # Maximum leaves per tree (controls model complexity)
    'max_depth': [5, 10, 15, 20, -1],  # Tree depth; -1 means no limit
    'min_child_samples': [10, 20, 50, 100, 200],  # Minimum data points in a child node
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],  # Row subsampling fraction
    'colsample_bytree': [0.4, 0.5, 0.6, 0.7, 0.8],  # Feature subsampling fraction
    'reg_alpha': [0.0, 0.1, 0.5, 1.0, 5.0],  # L1 regularization
    'reg_lambda': [0.0, 0.1, 0.5, 1.0, 5.0],  # L2 regularization
}

# Define a dictionary mapping pitch names to their respective DataFrames
pitch_dfs = {
    'Four-Seam': four_seam_df,
    'Sinker': sinker_df,
    'Cutter': cutter_df,
    'Slider': slider_df,
    'Sweeper': sweeper_df,
    'Curve': curve_df,
    'Offspeed': offspeed_df
}

# Dictionary to store the results for each pitch type
results = {}

for pitch_name, subset_df in pitch_dfs.items():
    print(f'Modeling {pitch_name}')
    
    # Call your training function for the current subset
    model, best_params, best_score = train_and_evaluate_with_cv(
        subset_df, features, target, param_grid, n_iter=20, cv=5
    )
    
    # Store the results in the dictionary
    results[pitch_name] = {
        'model': model,
        'best_params': best_params,
        'best_rmse': best_score
    }
    
    print(f"Modeled {pitch_name}") 
    print(f"Best RMSE for {pitch_name}: {best_score}")
    print(f"Best Parameters for {pitch_name}: {best_params}\n")

Modeling Four-Seam
Fitting 5 folds for each of 20 candidates, totalling 100 fits
[LightGBM] [Info] Total Bins 717
[LightGBM] [Info] Number of data points in the train set: 1031746, number of used features: 3
[LightGBM] [Info] Start training from score 11.043673
[LightGBM] [Info] Total Bins 717
[LightGBM] [Info] Number of data points in the train set: 1031746, number of used features: 3
[LightGBM] [Info] Start training from score 16.052961
Modeled Four-Seam
Best RMSE for Four-Seam: 3.3470213652726817
Best Parameters for Four-Seam: {'multioutputregressor__estimator__subsample': 0.8, 'multioutputregressor__estimator__reg_lambda': 1.0, 'multioutputregressor__estimator__reg_alpha': 0.1, 'multioutputregressor__estimator__num_leaves': 31, 'multioutputregressor__estimator__n_estimators': 2000, 'multioutputregressor__estimator__min_child_samples': 100, 'multioutputregressor__estimator__max_depth': -1, 'multioutputregressor__estimator__learning_rate': 0.01, 'multioutputregressor__estimator__cols